In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# ClusterDynamics – Unified Solver Notebook  [Windows / Nasr_Workstation]
# Ghoniem & Cho (1979) | 316 Stainless Steel | 450 °C | P = 1e-6 dpa/s
# Machine: Xeon Silver 4116 @ 2.10 GHz  |  12 cores / 24 threads
# Build:   MSVC 19.44  /O2 /arch:AVX2 /GL  +  MSVC OpenMP 2.0
# ══════════════════════════════════════════════════════════════════════════════
#
# Select solver mode by setting MODE below:
#
#   MODE                 Backend           NV (default)  NI (default)  Wall time
#   ─────────────────    ────────────────  ────────────  ────────────  ─────────
#   'py_segments'        Python LSODA            50           100       ~50 s
#   'py_full'            Python LSODA            50           100       ~17 s
#   'cpp_full'           CVODE full            100           1000       ~500 s
#   'cpp_expand_front'   CVODE Phase I         1,000        10,000      ~6 s
#   'cpp_dynamic_win'    CVODE Phase II        1,000       100,000      ~30 s
#   'cpp_sliding_win'    CVODE Phase III      10,000     1,000,000      ~78 s *
#   'sliding_OpenMP'     CVODE Phase IV       10,000     1,000,000      TBD
#
#   * Wall times are estimates from macOS M3 Max (-O3 -march=native).
#     Windows/MSVC (/O2 /arch:AVX2) timings will differ; run to measure.
#
# Phase descriptions:
#   py_segments        – scipy LSODA, segmented x_max-gated integration
#   py_full            – scipy LSODA, single solve_ivp call, no gating (reference)
#   cpp_full           – CVODE BDF/band, all Nv+Ni equations (window_mode=0, reference)
#   cpp_expand_front   – CVODE BDF/GMRES, dynamic upper-truncation window (Phase I)
#   cpp_dynamic_win    – Phase I + lower QSS contraction + geometric expansion (Phase II)
#   cpp_sliding_win    – constant-width W window slides with cluster front (Phase III)
#   sliding_OpenMP     – Phase III + OpenMP + /arch:AVX2 SIMD (Phase IV, Windows)
#

# ── 1. User selection ─────────────────────────────────────────────────────────
MODE = 'cpp_sliding_win'   # ← set desired solver mode here

# ── 2. Problem size (set to None to use the mode defaults in the table above) ─
NV = 1e4    # maximum vacancy cluster size
NI = 1e5    # maximum interstitial cluster size

# ── 3. Output and graph configuration ────────────────────────────────────────
SAVE_OUTPUT           = True   # save PNG figures to output/ directory
N_TIME_EVOL_CURVES    = 10     # time-evolution curves per panel (Figs 2–7)
N_SIZE_DIST_SNAPSHOTS = 10     # size-distribution snapshots per panel (Figs 8–13)

# ── 3a. Run identifier tags (optional) ───────────────────────────────────────
# Extra labels appended to the output directory name, separated by '_'.
# Example: TAGS = ['T450', 'P1e-6']  →  ...output/20260316_182017_py_full_T450_P1e-6/
# Leave as an empty list for the default naming (timestamp + mode only).
TAGS = ['T450','Nv1e4_Ni1e5','P(1e-3)','Z_i(1.08)', 'rho_d(1e15)']

# ── 4. Material / physics overrides ──────────────────────────────────────────
# Any key from InputData.material_params can be overridden here.
# Unspecified keys keep their Ghoniem & Cho (1979) defaults.
#
# Example:
#   PHYSICS = {'T': 773, 'P': 1e-5, 'rho_d': 1e10}   # 500 °C, 10× dose rate
PHYSICS = {'T':273+450, 'P':1e-3}

# ── 5. Solver overrides (optional) ───────────────────────────────────────────
# Top-level keys (t_span, n_points, rtol, atol, n_segments, log_time) are
# merged into SOLVER_CONFIG.  Keys that belong inside solver_method (linsol,
# lmm, window_mode, window_w0_v, window_w0_i, window_C_expand, window_width,
# window_t_start, window_prec, window_omp_threads, …) are forwarded there.
#
# Examples:
#   KWARGS = {'rtol': 1e-6, 'atol': 1e-25}                    # tighter tolerances
#   KWARGS = {'t_span': (1e-5, 1e7), 'n_points': 500}         # longer run, fewer points
#   KWARGS = {'linsol': 'gmres'}                               # cpp_full with GMRES
#   KWARGS = {'window_omp_threads': 6}                         # fewer OMP threads
KWARGS = {'t_span': (1e-6, 1e5)}

# ═════════════════════════════════════════════════════════════════════════════
# Nothing below this line needs to be changed for a standard run.
# ═════════════════════════════════════════════════════════════════════════════

import sys, time, warnings
import numpy as np
from pathlib import Path
from scipy.integrate import solve_ivp
warnings.filterwarnings('ignore')

# ── Path setup ────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = _NOTEBOOK_DIR.parent      # ClusterDynamics/
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from py_utils.simulation     import ClusterDynamicsSimulation
from py_utils.reaction_rates import ReactionRates
from py_utils.rate_equations import RateEquations
from py_utils.cpp_bridge     import run_cpp_solver
from py_utils.visualization  import plot_results
from py_utils                import pre_process, post_process

# ── Validate mode and apply defaults ─────────────────────────────────────────
_VALID_MODES = ('py_segments', 'py_full', 'cpp_full', 'cpp_expand_front',
                'cpp_dynamic_win', 'cpp_sliding_win', 'sliding_OpenMP')
if MODE not in _VALID_MODES:
    raise ValueError(f'MODE={MODE!r} is not valid. Choose from: {_VALID_MODES}')

_DEFAULTS = {
    'py_segments':       {'NV':    50, 'NI':      100},
    'py_full':           {'NV':    50, 'NI':      100},
    'cpp_full':          {'NV':   100, 'NI':    1000},
    'cpp_expand_front':  {'NV':  1000, 'NI':    10000},
    'cpp_dynamic_win':   {'NV':  1000, 'NI':   100000},
    'cpp_sliding_win':   {'NV':  1000, 'NI':   100000},
    'sliding_OpenMP':    {'NV': 10000, 'NI':  1000000},
}
NV = NV or _DEFAULTS[MODE]['NV']
NI = NI or _DEFAULTS[MODE]['NI']

print('=' * 65)
print(f'ClusterDynamics  |  MODE = "{MODE}"')
print(f'Ghoniem & Cho (1979) | 316 SS | 450 °C | P=1e-6 dpa/s')
print(f'Problem size: Nv={NV:,}, Ni={NI:,}, N_EQ={NV+NI:,}')
print('=' * 65)

# ── Initialise simulation ─────────────────────────────────────────────────────
sim = ClusterDynamicsSimulation(Nv=NV, Ni=NI)

# ── Apply PHYSICS overrides and rebuild derived quantities ────────────────────
if PHYSICS:
    _invalid = set(PHYSICS) - set(sim.input_data.material_params)
    if _invalid:
        raise ValueError(f'Unknown PHYSICS keys: {_invalid}. '
                         f'Valid keys: {set(sim.input_data.material_params)}')
    sim.input_data.material_params.update(PHYSICS)
    sim.input_data.calculate_derived_parameters()
    sim.reaction_rates = ReactionRates(sim.input_data)
    sim.rate_equations = RateEquations(sim.input_data, sim.reaction_rates)
    print(f'PHYSICS applied: {PHYSICS}')

t0 = time.perf_counter()

# ── Solver dispatch ───────────────────────────────────────────────────────────
if MODE == 'py_segments':
    SOLVER_CONFIG = {
        't_span':     (1e-6, 1e6),
        'n_segments': 60,
        'rtol':       1e-5,
        'atol':       1e-20,
    }
    LABEL = 'Python LSODA (segmented, x_max-gated)'

elif MODE == 'py_full':
    # Full system – single solve_ivp call, no x_max gating, no segmentation.
    # Integrates all Nv+Ni equations in one shot via scipy LSODA.
    # Use as the Python accuracy reference for small problems.
    SOLVER_CONFIG = {
        't_span': (1e-6, 1e6),
        'rtol':   1e-5,
        'atol':   1e-20,
    }
    LABEL = 'Python LSODA full system (no gating)'

elif MODE == 'cpp_full':
    # Full system – CVODE BDF/band, window_mode=0 (no truncation).
    # Integrates all Nv+Ni equations simultaneously.
    # Use as the accuracy reference for benchmarking windowed modes.
    SOLVER_CONFIG = {
        't_span':   (1e-6, 1e6),
        'n_points': 200,
        'rtol':     1e-5,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':     'cvode',
            'lmm':         'bdf',
            'linsol':      'band',
            'window_mode': 0,
        },
    }
    LABEL = 'C++ CVODE BDF full system (window_mode=0)'

elif MODE == 'cpp_expand_front':
    # Phase I – dynamic upper-truncation window
    # Starts with a small active window and expands only when needed.
    # ~40× speedup vs cpp_full for Ni=10,000.
    SOLVER_CONFIG = {
        't_span':   (1e-6, 1e6),
        'n_points': 200,
        'rtol':     1e-5,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':            'cvode',
            'lmm':                'bdf',
            'linsol':             'gmres',
            'window_mode':        1,
            'window_w0_v':        100,
            'window_w0_i':        100,
            'window_C_expand':    1e-20,
            'window_expand_pad':  20,
            'window_expand_factor': 2.0,   # geometric doubling: ~7 reinits vs ~495
            'window_check_every': 1,
        },
    }
    LABEL = 'C++ CVODE BDF Phase I (expanding front)'

elif MODE == 'cpp_dynamic_win':
    # Phase II – sliding window with lower QSS contraction
    # Upper expansion (geometric) + lower contraction (QSS criterion).
    # Jacobi diagonal preconditioner for GMRES. ~0.01% accuracy vs cpp_full.
    SOLVER_CONFIG = {
        't_span':   (1e-6, 1e6),
        'n_points': 200,
        'rtol':     1e-4,
        'atol':     1e-18,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          2,
            'window_w0_v':          100,
            'window_w0_i':          100,
            'window_C_expand':      1e-20,
            'window_expand_pad':    20,
            'window_expand_factor': 2.0,
            'window_check_every':   20,
            'window_C_contract':    1e-3,
            'window_min_active_i':  100,
            'window_prec':          1,
        },
    }
    LABEL = 'C++ CVODE BDF Phase II (dynamic window)'

elif MODE == 'cpp_sliding_win':
    # Phase III – constant-width sliding window
    # Fixed-width window W slides rigidly with the cluster front.
    # Nucleation guard at t < window_t_start prevents early lower-front sliding.
    # Scales to Ni = 1,000,000.
    # SIMD speedup from /arch:AVX2 (AVX2 + FMA, Xeon Silver 4116 / Skylake-SP).
    #
    # window_expand_factor=1.0 + window_expand_pad=W ensures the window always
    # advances by exactly W at each expansion event, keeping N_active = W+1
    # constant (500 Cv + 1000 Ci).  expand_factor=2.0 caused doubling and an
    # 80× slowdown on the prior benchmark run.
    SOLVER_CONFIG = {
        't_span':   (1e-6, 1e6),
        'n_points': 200,
        'rtol':     1e-4,
        'atol':     1e-18,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          3,
            'window_w0_v':          500,
            'window_w0_i':          1000,
            'window_C_expand':      1e-18,
            'window_expand_pad':    1000,   # = window_width → advance by exactly W
            'window_expand_factor': 1.0,    # disable geometric doubling
            'window_check_every':   10,
            'window_width':         1000,
            'window_t_start':       10.0,
            'window_N_thresh':      2000,
            'window_prec':          1,
        },
    }
    LABEL = 'C++ CVODE BDF Phase III (sliding window, Windows/AVX2)'

else:  # 'sliding_OpenMP'
    # Phase IV – sliding window + OpenMP + SIMD  [Windows / Nasr_Workstation]
    #
    # Three speedup layers:
    #   1. /arch:AVX2 /GL (MSVC)  — AVX2 + FMA SIMD auto-vectorisation +
    #                               whole-program optimisation (link-time codegen)
    #   2. Pre-allocated scratch buffers (Cv_buf / Ci_buf) — eliminates heap
    #      allocation from every CVODE RHS call (millions per run)
    #   3. Single #pragma omp parallel region per RHS call — one fork-join,
    #      not many; OMP_MIN_WORK=20000 threshold auto-falls back to serial
    #      path when W < 20000 (current W=1000 uses serial + buffer path only)
    #
    # Hardware: Xeon Silver 4116  12 physical cores / 24 logical processors
    #   window_omp_threads = 12  (physical cores; HT does not help FP-heavy loops)
    #
    # window_expand_factor=1.0 + window_expand_pad=W: see cpp_sliding_win note.
    #
    # NOTE: /fp:fast (/fp:precise is used instead) — intentionally omitted.
    #   Fast-math alters IEEE-754 semantics and can cause CVODE's stiff-error
    #   estimator to diverge.
    SOLVER_CONFIG = {
        't_span':   (1e-6, 1e6),
        'n_points': 200,
        'rtol':     1e-4,
        'atol':     1e-18,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          4,
            'window_omp_threads':   12,    # 12 physical cores (Xeon Silver 4116)
            'window_w0_v':          500,
            'window_w0_i':          1000,
            'window_C_expand':      1e-18,
            'window_expand_pad':    1000,   # = window_width → advance by exactly W
            'window_expand_factor': 1.0,    # disable geometric doubling
            'window_check_every':   10,
            'window_width':         1000,
            'window_t_start':       10.0,
            'window_N_thresh':      2000,
            'window_prec':          1,
        },
    }
    LABEL = 'C++ CVODE BDF Phase IV (sliding window + OpenMP, Windows/AVX2)'

# ── Apply KWARGS overrides ────────────────────────────────────────────────────
_solver_method_keys = set(SOLVER_CONFIG.get('solver_method', {}).keys())
for _k, _v in KWARGS.items():
    if _k in SOLVER_CONFIG:
        SOLVER_CONFIG[_k] = _v
    elif _k in _solver_method_keys or _k.startswith('window_') or _k in (
            'backend', 'lmm', 'linsol', 'ark_table'):
        SOLVER_CONFIG.setdefault('solver_method', {})[_k] = _v
    else:
        SOLVER_CONFIG[_k] = _v
if KWARGS:
    print(f'KWARGS applied: {KWARGS}')

# Stamp the mode name so the output directory and provenance file match MODE.
SOLVER_CONFIG['mode'] = MODE

# ── Run ───────────────────────────────────────────────────────────────────────
if MODE == 'py_segments':
    results = sim.run_simulation(
        t_span     = SOLVER_CONFIG['t_span'],
        n_segments = SOLVER_CONFIG.get('n_segments', 60),
        rtol       = SOLVER_CONFIG.get('rtol', 1e-5),
        atol       = SOLVER_CONFIG.get('atol', 1e-20),
        verbose    = True,
    )

elif MODE == 'py_full':
    print(f"\nRunning Python LSODA full system  (no gating,"
          f"  t ∈ [{SOLVER_CONFIG['t_span'][0]:.1e},"
          f" {SOLVER_CONFIG['t_span'][1]:.1e}] s)")
    y0  = pre_process.get_initial_conditions(sim.rate_equations)
    sol = solve_ivp(
        sim.rate_equations.ode_system,
        SOLVER_CONFIG['t_span'], y0,
        method = 'LSODA',
        rtol   = SOLVER_CONFIG.get('rtol', 1e-5),
        atol   = SOLVER_CONFIG.get('atol', 1e-20),
        dense_output = False,
    )
    if not sol.success:
        print(f'  ⚠️  solve_ivp did not converge: {sol.message}')
    print(f'  nfev={sol.nfev}  njev={sol.njev}  status={sol.status}')
    results = post_process.calculate_derived_quantities(
        sol.t, sol.y, sim.input_data, sim.rate_equations,
    )
    results['metadata'] = {
        'solver_stats': {
            'success':       sol.success,
            'message':       'Python LSODA full system',
            'wall_time':     time.perf_counter() - t0,
            'n_time_points': sol.t.size,
        },
    }

else:
    results = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)

elapsed = time.perf_counter() - t0
print(f'\nWall time: {elapsed:.1f} s')

# ── Plot configuration ────────────────────────────────────────────────────────
PLOT_CONFIG = {
    'point_defects': {
        'xlim': (1e-6, 1e7),
        'ylim': (1e-15, 1e-3),
    },
    'interstitial_clusters': {
        'xlim': {
            'small': (1e-6, 1e5),
            'mid':   (1e2,  1e7),
            'large': (1e2,  1e7),
        },
        'C_floor':  1e-25,
        'n_curves': N_TIME_EVOL_CURVES,
    },
    'vacancy_clusters': {
        'xlim': {
            'small': (1e-6, 1e5),
            'mid':   (1e2,  1e7),
            'large': (1e2,  1e7),
        },
        'C_floor':  1e-20,
        'n_curves': N_TIME_EVOL_CURVES,
    },
    'i_cluster_size': {
        'xlim':        None,
        'ylim':        None,
        'n_snapshots': N_SIZE_DIST_SNAPSHOTS,
    },
    'v_cluster_size': {
        'xlim':        None,
        'ylim':        None,
        'n_snapshots': N_SIZE_DIST_SNAPSHOTS,
    },
}

# ── Plots ─────────────────────────────────────────────────────────────────────
if results is not None:
    run_dir = plot_results(
        sim, results,
        output_dir  = str(OUTPUT_DIR),
        save_plots  = SAVE_OUTPUT,
        sim_config  = SOLVER_CONFIG,
        label       = LABEL,
        plot_config = PLOT_CONFIG,
        wall_time   = elapsed,
        tags        = TAGS or None,
    )

# ── Summary diagnostics ───────────────────────────────────────────────────────
if results is not None:
    conc  = results['concentrations']
    t     = results['time']
    T_run = sim.input_data.material_params['T']
    P_run = sim.input_data.material_params['P']
    print('\n' + '=' * 65)
    print('SUMMARY')
    print('=' * 65)
    print(f'  Mode:         {MODE}')
    print(f'  T / P:        {T_run} K  /  {P_run:.1e} dpa s^-1')
    print(f'  Nv / Ni:      {NV:,} / {NI:,}  (N_EQ = {NV+NI:,})')
    print(f'  t_final:      {t[-1]:.2e} s')
    print(f'  Wall time:    {elapsed:.1f} s')
    print(f'  Cv1(t_end):   {conc["Cv1"][-1]:.4e}')
    print(f'  Ci1(t_end):   {conc["Ci1"][-1]:.4e}')
    if MODE not in ('py_segments', 'py_full') and 'active_band' in results:
        active = results['active_band']
        lo_i = int(active['x_min'][-1])
        hi_i = int(active['x_max'][-1])
        if hi_i > 0:
            print(f'  Active Ci:    [{lo_i}..{hi_i}]  ({hi_i/NI*100:.1f}% of Ni)')
        hi_v = next((x for x in range(int(NV), 0, -1)
                     if conc.get(f'Cv{x}', [0.0])[-1] > 0.0), 0)
        if hi_v > 0:
            print(f'  Active Cv:    [1..{hi_v}]')
    print(f'  Graphs:       {N_TIME_EVOL_CURVES} time-evol curves/panel  |  '
          f'{N_SIZE_DIST_SNAPSHOTS} size-dist snapshots/panel')
    if SAVE_OUTPUT and 'run_dir' in dir():
        print(f'  Output:       {run_dir}')
    print('=' * 65)
else:
    print(f'\n  Simulation FAILED for MODE="{MODE}"')

ClusterDynamics  |  MODE = "cpp_sliding_win"
Ghoniem & Cho (1979) | 316 SS | 450 °C | P=1e-6 dpa/s
Problem size: Nv=10,000.0, Ni=100,000.0, N_EQ=110,000.0
Initializing ClusterDynamics simulation…
Loading parameters from: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Full_CD\input\input_parameters.xlsx
Successfully loaded all three parameter sheets.
Derived:  T=723.0 K  Cv_eq=7.030e-12  alpha=9.685e+12  Di=2.659e-08 m^2/s  Dv=1.148e-15 m^2/s
ReactionRates: KCV[0]=1.735e+04  KLI[0]=3.388e+11  GCV[0]=2.646e+00
RateEquations: N=110000 equations  (Nv=10000, Ni=100000)
✓ Setup validation complete.
✓ Simulation initialized successfully!
Derived:  T=723.0 K  Cv_eq=7.030e-12  alpha=9.685e+12  Di=2.659e-08 m^2/s  Dv=1.148e-15 m^2/s
ReactionRates: KCV[0]=1.735e+04  KLI[0]=3.388e+11  GCV[0]=2.646e+00
RateEquations: N=110000 equations  (Nv=10000, Ni=100000)
PHYSICS applied: {'T': 723, 'P': 0.001}
KWARGS applied: {'t_span': (1e-06, 100000.0)}
Running C++ solver (solver.exe)  …  (Nv=10000, Ni=

findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeOneSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeTwoSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeThreeSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFourSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFiveSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmsy10'] not

  ✓ point_defects.png
  ✓ interstitial_clusters_small.png
  ✓ interstitial_clusters_mid.png
  ✓ interstitial_clusters_large.png
  ✓ vacancy_clusters_small.png
  ✓ vacancy_clusters_mid.png
  ✓ vacancy_clusters_large.png
  ✓ i_cluster_size_early.png
  ✓ i_cluster_size_mid.png
  ✓ i_cluster_size_late.png
  ✓ v_cluster_size_early.png
  ✓ v_cluster_size_mid.png
  ✓ v_cluster_size_late.png
  ✓ i_cluster_radius_small.png
  ✓ i_cluster_radius_mid.png
  ✓ i_cluster_radius_large.png
  ✓ 20260321_122359_cpp_sliding_win.md

SUMMARY
  Mode:         cpp_sliding_win
  T / P:        723 K  /  1.0e-03 dpa s^-1
  Nv / Ni:      10,000.0 / 100,000.0  (N_EQ = 110,000.0)
  t_final:      1.00e+05 s
  Wall time:    912.7 s
  Cv1(t_end):   3.8883e-05
  Ci1(t_end):   2.4673e-12
  Active Ci:    [1..6000]  (6.0% of Ni)
  Active Cv:    [1..63]
  Graphs:       10 time-evol curves/panel  |  10 size-dist snapshots/panel
  Output:       C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Full_CD\output\20260321_12235

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETER OPTIMIZATION  –  sliding_window mode  |  NV=1000  NI=1e5
#
# Find combinations of (E_m_i, E_b_2i, Z_i, rho_d) that produce:
#   • Interstitial cluster number density : 1e19 – 1e20  m^-3
#   • Mean interstitial cluster diameter  : 5  – 30      nm
#
# Strategy: scipy differential_evolution on a log-penalised objective.
#   popsize=5  →  initial population of 4×5=20 evaluations
#   maxiter=8  →  up to ~60 total solver calls  (~30 min at ~30 s/call)
# ══════════════════════════════════════════════════════════════════════════════

import sys, io, time, warnings
import numpy as np
from scipy.optimize import differential_evolution
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Path setup ────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path().resolve()

BASE_DIR = _NOTEBOOK_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from py_utils.simulation    import ClusterDynamicsSimulation
from py_utils.reaction_rates import ReactionRates
from py_utils.rate_equations import RateEquations
from py_utils.cpp_bridge    import run_cpp_solver


# ══════════════════════════════════════════════════════════════════════════════
# 1. PROBLEM SIZE AND PHYSICAL TARGETS
# ══════════════════════════════════════════════════════════════════════════════

NV_OPT = 1000
NI_OPT = int(1e5)

TARGET_DENSITY_MIN = 1e21   # interstitial cluster density  [m^-3]
TARGET_DENSITY_MAX = 1e22
TARGET_DIAM_MIN    = 2.0    # mean cluster diameter [nm]
TARGET_DIAM_MAX    = 5.0

# ── Unit-conversion helpers ───────────────────────────────────────────────────
_a_nm   = 0.363                        # lattice parameter [nm]  (3.63 Å)
_n_atom = 4.0 / (_a_nm * 1e-9)**3     # FCC atomic number density [m^-3]

def _mean_diam_nm(mean_n):
    """
    Mean diameter [nm] of a circular Frank partial loop containing mean_n SIA.
    Burgers vector b = a/sqrt(3), loop area = n*Omega/b, Omega = a^3/4
      => r [nm] = a_nm * sqrt( n*sqrt(3) / (4*pi) )
    """
    return 2.0 * _a_nm * np.sqrt(mean_n * np.sqrt(3.0) / (4.0 * np.pi))

def _cluster_density_m3(total_i_frac, mean_n):
    """
    Convert atom-fraction totals to cluster number density [m^-3].
      total_i_frac = sum_x( x * Ci_x )   [dimensionless atom fraction]
      mean_n       = sum_x(x*Ci_x) / sum_x(Ci_x)
    => N_clusters/vol = (total_i_frac / mean_n) * n_atom
    """
    return (total_i_frac / max(mean_n, 1.0)) * _n_atom


# ══════════════════════════════════════════════════════════════════════════════
# 2. SLIDING-WINDOW SOLVER CONFIGURATION  (tuned for NV=1000, NI=1e5)
# ══════════════════════════════════════════════════════════════════════════════

OPT_SOLVER_CONFIG = {
    't_span':   (1e-5, 1e7),
    'n_points': 200,           # fewer output rows → faster I/O
    'rtol':     1e-4,
    'atol':     1e-18,
    'log_time': True,
    'solver_method': {
        'backend':              'cvode',
        'lmm':                  'bdf',
        'linsol':               'gmres',
        'window_mode':          3,      # Phase III – constant-width sliding window
        'window_w0_v':          200,
        'window_w0_i':          500,
        'window_C_expand':      1e-18,
        'window_expand_pad':    500,    # = window_width -> advance by exactly W
        'window_expand_factor': 1.0,    # disable geometric doubling
        'window_check_every':   10,
        'window_width':         500,
        'window_t_start':       10.0,
        'window_N_thresh':      2000,
        'window_prec':          1,
    },
}


# ══════════════════════════════════════════════════════════════════════════════
# 3. PARAMETER SEARCH BOUNDS
#    rho_d is searched on a log10 scale to cover several decades uniformly.
# ══════════════════════════════════════════════════════════════════════════════

PARAM_BOUNDS = {
    'E_m_i':      (0.02, 1.0),          # [eV]
    'E_b_2i':     (0.70, 1.5),          # [eV]
    'Z_i':        (1.02, 1.10),          # [-]
    'log10_rho_d': (14.0, 15.0),          # log10 of rho_d [m^-2]
}

_BOUNDS_LIST = [
    PARAM_BOUNDS['E_m_i'],
    PARAM_BOUNDS['E_b_2i'],
    PARAM_BOUNDS['Z_i'],
    PARAM_BOUNDS['log10_rho_d'],
]


# ══════════════════════════════════════════════════════════════════════════════
# 4. SINGLE-EVALUATION RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def _run_one(E_m_i, E_b_2i, Z_i, rho_d):
    """
    Build a simulation with the given material parameters, run the C++ solver,
    and return (density [m^-3], mean_diam [nm]).
    Returns (None, None) on solver failure.
    """
    # Redirect noisy init printout to /dev/null during optimization
    _old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        sim = ClusterDynamicsSimulation(Nv=NV_OPT, Ni=NI_OPT)
    finally:
        sys.stdout = _old_stdout

    # Override material parameters and rebuild all derived quantities
    sim.input_data.material_params['E_m_i']  = float(E_m_i)
    sim.input_data.material_params['E_b_2i'] = float(E_b_2i)
    sim.input_data.material_params['Z_i']    = float(Z_i)
    sim.input_data.material_params['rho_d']  = float(rho_d)

    _old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        sim.input_data.calculate_derived_parameters()
        sim.reaction_rates = ReactionRates(sim.input_data)
        sim.rate_equations = RateEquations(sim.input_data, sim.reaction_rates)
        results = run_cpp_solver(sim, OPT_SOLVER_CONFIG, base_dir=BASE_DIR)
    finally:
        sys.stdout = _old_stdout

    if results is None:
        return None, None

    mean_n  = results['mean_sizes']['mean_i'][-1]    # interstitials / cluster
    total_i = results['totals']['total_i'][-1]        # atom fraction (all i-clusters)

    density = _cluster_density_m3(total_i, mean_n)
    diam    = _mean_diam_nm(mean_n)
    return density, diam


# ══════════════════════════════════════════════════════════════════════════════
# 5. OBJECTIVE FUNCTION  (penalty = 0 inside both target windows)
# ══════════════════════════════════════════════════════════════════════════════

_eval_log = []   # accumulates every evaluation for post-analysis

def _penalty(density, diam):
    """
    Log-scale penalty for density; linear-scale penalty for diameter.
    Both terms are zero while inside the target range.
    """
    if density is None or density <= 0:
        return 1e6

    log_d  = np.log10(density)
    log_lo = np.log10(TARGET_DENSITY_MIN)
    log_hi = np.log10(TARGET_DENSITY_MAX)
    p_dens = max(log_lo - log_d, 0.0)**2 + max(log_d - log_hi, 0.0)**2

    # Scale factor keeps both penalty components order-O(1) at 1-decade miss
    p_diam = (max(TARGET_DIAM_MIN - diam, 0.0)**2 +
              max(diam - TARGET_DIAM_MAX, 0.0)**2) * 0.01

    return p_dens + p_diam


def _objective(x):
    E_m_i, E_b_2i, Z_i, log_rho = x
    rho_d = 10.0**log_rho

    t0 = time.perf_counter()
    density, diam = _run_one(E_m_i, E_b_2i, Z_i, rho_d)
    elapsed = time.perf_counter() - t0

    pen = _penalty(density, diam)

    record = dict(
        E_m_i=E_m_i, E_b_2i=E_b_2i, Z_i=Z_i, rho_d=rho_d,
        density=density, diam_nm=diam,
        penalty=pen, wall_s=elapsed,
    )
    _eval_log.append(record)

    # Progress line
    n      = len(_eval_log)
    d_str  = f'{density:.2e}' if density else 'FAIL'
    dm_str = f'{diam:.1f}'    if diam    else 'N/A'
    flag   = 'IN RANGE' if pen == 0.0 else f'pen={pen:.3f}'
    print(f"  [{n:3d}] E_mi={E_m_i:.3f} E_b2i={E_b_2i:.3f} "
          f"Zi={Z_i:.3f} log_rho={log_rho:.2f}  ->  "
          f"rho_i={d_str} m^-3  d={dm_str} nm  {flag}  ({elapsed:.1f}s)")

    return pen


# ══════════════════════════════════════════════════════════════════════════════
# 6. RUN DIFFERENTIAL EVOLUTION
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 75)
print('PARAMETER OPTIMIZATION  |  sliding_window  |  NV=1000  NI=1e5')
print(f'Targets : rho_i in [1e21, 1e22] m^-3   mean_diam in [2, 5] nm')
print(f'Search  : E_m_i    in {PARAM_BOUNDS["E_m_i"]} eV')
print(f'          E_b_2i   in {PARAM_BOUNDS["E_b_2i"]} eV')
print(f'          Z_i      in {PARAM_BOUNDS["Z_i"]}')
print(f'          log10(rho_d) in {PARAM_BOUNDS["log10_rho_d"]}  [cm^-2]')
print('Optimizer: differential_evolution  popsize=5  maxiter=8')
print('=' * 75)

_eval_log.clear()
t_opt_start = time.perf_counter()

opt_result = differential_evolution(
    _objective,
    bounds   = _BOUNDS_LIST,
    strategy = 'best1bin',
    maxiter  = 8,
    popsize  = 5,      # 4 params * 5 = 20 initial population
    tol      = 0.01,   # stop early if std(population penalties) < tol * mean
    seed     = 42,
    workers  = 1,      # must be serial – CVODE subprocess is not fork-safe
    disp     = False,
    polish   = False,
)

t_opt_total = time.perf_counter() - t_opt_start


# ══════════════════════════════════════════════════════════════════════════════
# 7. RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 75)
print('OPTIMIZATION COMPLETE')
print(f'  Total evaluations : {len(_eval_log)}')
print(f'  Total wall time   : {t_opt_total/60:.1f} min')
print(f'  Final penalty     : {opt_result.fun:.4f}')
print('=' * 75)

best_evals = sorted(_eval_log, key=lambda r: r['penalty'])

print('\nTop 5 parameter sets (sorted by penalty):')
hdr = (f"{'#':>3}  {'E_m_i':>7}  {'E_b_2i':>7}  {'Z_i':>6}  "
       f"{'log_rho':>8}  {'rho_i [m-3]':>12}  {'d [nm]':>8}  {'penalty':>8}")
print(hdr)
print('-' * len(hdr))
for rank, r in enumerate(best_evals[:5], 1):
    d_str  = f'{r["density"]:.2e}' if r['density'] else 'FAIL'
    dm_str = f'{r["diam_nm"]:.1f}' if r['diam_nm'] else 'N/A'
    print(f"{rank:3d}  {r['E_m_i']:7.4f}  {r['E_b_2i']:7.4f}  {r['Z_i']:6.4f}  "
          f"{np.log10(r['rho_d']):8.3f}  {d_str:>12}  {dm_str:>8}  {r['penalty']:8.4f}")

best = best_evals[0]
print()
if best['penalty'] == 0.0:
    print('TARGET ACHIEVED. Best parameters:')
else:
    print('Best parameters found (targets partially met):')

print(f"  E_m_i  = {best['E_m_i']:.4f} eV          (baseline: 0.9 eV)")
print(f"  E_b_2i = {best['E_b_2i']:.4f} eV          (baseline: 0.4 eV)")
print(f"  Z_i    = {best['Z_i']:.4f}                (baseline: 1.08)")
print(f"  rho_d  = {best['rho_d']:.3e} cm^-2    (baseline: 1e9)")
print()
d_in  = (TARGET_DENSITY_MIN <= best['density'] <= TARGET_DENSITY_MAX) if best['density'] else False
dm_in = (TARGET_DIAM_MIN    <= best['diam_nm'] <= TARGET_DIAM_MAX)    if best['diam_nm']  else False
print(f"  => rho_i = {best['density']:.2e} m^-3   {'[IN RANGE]' if d_in else '[OUT OF RANGE]'}"
      f"   target: [1e19, 1e20]")
print(f"  => d_mean= {best['diam_nm']:.1f} nm          {'[IN RANGE]' if dm_in else '[OUT OF RANGE]'}"
      f"   target: [5, 30] nm")


PARAMETER OPTIMIZATION  |  sliding_window  |  NV=1000  NI=1e5
Targets : rho_i in [1e21, 1e22] m^-3   mean_diam in [2, 5] nm
Search  : E_m_i    in (0.02, 1.0) eV
          E_b_2i   in (0.7, 1.5) eV
          Z_i      in (1.02, 1.1)
          log10(rho_d) in (14.0, 15.0)  [cm^-2]
Optimizer: differential_evolution  popsize=5  maxiter=8
  [  1] E_mi=0.774 E_b2i=1.167 Zi=1.091 log_rho=14.21  ->  rho_i=7.32e+16 m^-3  d=0.4 nm  pen=17.130  (840.1s)
  [  2] E_mi=0.208 E_b2i=1.362 Zi=1.065 log_rho=14.47  ->  rho_i=2.11e+18 m^-3  d=2.7 nm  pen=7.162  (171.5s)
  [  3] E_mi=0.516 E_b2i=0.746 Zi=1.050 log_rho=14.09  ->  rho_i=4.78e+16 m^-3  d=0.4 nm  pen=18.696  (650.2s)
  [  4] E_mi=0.902 E_b2i=0.906 Zi=1.074 log_rho=14.77  ->  rho_i=4.60e+16 m^-3  d=0.4 nm  pen=18.840  (764.1s)
  [  5] E_mi=0.710 E_b2i=0.828 Zi=1.023 log_rho=14.36  ->  rho_i=5.92e+16 m^-3  d=0.4 nm  pen=17.901  (824.9s)
  [  6] E_mi=0.635 E_b2i=1.419 Zi=1.059 log_rho=14.27  ->  rho_i=7.28e+16 m^-3  d=0.4 nm  pen=17.147  (1318.7s)